# Word Embeddings and Semantic Similarity

**Author**: Tianxiang (Adam) Gao  
**Course**: CSC 383 / 483 – Applied Deep Learning  
**Description**:

In this assignment, you will explore word embeddings—dense vector representations of words that capture semantic meaning—by both training embeddings and analyzing pre-trained models.

You will implement a Skip-gram word embedding model on a small toy dataset to understand how embeddings are learned from local context. You will then use a pre-trained embedding matrix to examine semantic relationships, including nearest neighbors and word analogy analysis in the embedding space.

## Setup: Libraries

We will first import the main libraries used throughout this assignment:

- `tensorflow` / `keras`: for building and training deep learning models.  
- `keras.layers`: provides essential components such as convolution, pooling, and upsampling layers.  
- `tensorflow_datasets (tfds)`: to load the Oxford-IIIT Pet dataset easily.  
- `matplotlib`: for visualizing images, masks, and predictions.  
- `numpy`: for numerical operations and array manipulation.  
- `re` (regular expressions): text cleaning and tokenizes raw text into word-level tokens.
- `collections.Counter`: counts word frequencies to build the vocabulary.


In [1]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
import numpy as np
import re
from collections import Counter

2026-03-05 22:19:34.096228: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Using GPU

Keras automatically uses available GPUs when training, so no extra configuration is needed.  
You can verify that your GPU is detected with the following command:


In [2]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'),
 PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]

## Global Configuration

Below are the **global variables** that define key settings for this assignment, including batch size, learning rate, etc.

> **Note**: You can adjust these values to experiment with training speed,
model size, or accuracy.


In [3]:
MAX_VOCAB = 20_000     # maximum vocabulary size
MIN_FREQ = 2           # minimum token frequency
WINDOW = 4             # context window size
NEG_K = 5              # number of negative samples
BATCH_SIZE = 2048
EMB_DIM = 128           # embedding dimension
LEARNING_RATE = 1e-3
EPOCHS = 20

print("Configuration:")
print(f"  Max vocab size: {MAX_VOCAB}")
print(f"  Min frequency: {MIN_FREQ}")
print(f"  Window size: {WINDOW}")
print(f"  Negative samples: {NEG_K}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Embedding dimension: {EMB_DIM}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {EPOCHS}")

Configuration:
  Max vocab size: 20000
  Min frequency: 2
  Window size: 4
  Negative samples: 5
  Batch size: 2048
  Embedding dimension: 128
  Learning rate: 0.001
  Epochs: 20


## Dataset & Vocabulary

We first download and read the [TinyShakespeare](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt). Then build vocabulary using the distinct chars. With a specified sequence length, we build the source sequences and target sequences for training.


**1.** Download and read a toy text dataset, [TinyShakespeare](https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt), which comprises approximately 40,000 lines from various works by William Shakespeare.

In [4]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-03-05 22:19:37--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.01s   

2026-03-05 22:19:37 (73.3 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [5]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [6]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


**2.** In this step, we convert the raw text into *tokens (words)* in order to build a vocabulary. We preprocess the text by lowercasing it and using regular expressions (`re`) to keep only letters and apostrophes (`'`), while replacing all other characters with spaces.

In [7]:
def clean_text(text): return re.sub(r"[^a-zA-Z']+", " ", text.lower())

In [8]:
# Lowercase, keep letters/apostrophes, replace everything else with space
clean = clean_text(text)
tokens = clean.split()

In [9]:
print(clean[:100])
print("Num tokens:", len(tokens))
print("Sample:", tokens[:10])

first citizen before we proceed any further hear me speak all speak speak first citizen you are all 
Num tokens: 204062
Sample: ['first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak']


**3.** Once we obtain the cleaned text and split it into `tokens`, we build a vocabulary using a frequency count. Specifically, we use `Counter` to count how often each token appears in the dataset, keep only tokens whose frequency is at least `MIN_FREQ`, and then sort the remaining tokens by frequency in descending order. Finally, we limit the vocabulary size to the top `MAX_VOCAB` most frequent tokens to improve computational efficiency and ensure more stable training.

In [10]:
counts = Counter(tokens)
list(counts.items())[:10]

[('first', 362),
 ('citizen', 100),
 ('before', 195),
 ('we', 862),
 ('proceed', 21),
 ('any', 189),
 ('further', 36),
 ('hear', 230),
 ('me', 1769),
 ('speak', 293)]

In [11]:
vocab = [w for w in counts if counts[w] >= MIN_FREQ]
vocab = sorted(vocab, key=lambda w: counts[w], reverse=True)[:MAX_VOCAB]

**4.** Next, we construct two lookup mappings for converting between words and integer IDs:

- `s2i` (*string-to-index*): maps each word in the vocabulary to a unique integer ID.
- `i2s` (*index-to-string*): performs the reverse mapping from integer IDs back to words.

We reserve index `0` for a special token `<UNK>` (unknown), which is used to represent any word that is not in the vocabulary. This allows the model to handle unseen or rare words gracefully. We then assign integer IDs to all vocabulary words starting from index `1`, and finally build `i2s` by reversing the `s2i` mapping.

Since `<UNK>` is part of the embedding space and may appear during encoding, we define the vocabulary size as `vocab_size = len(s2i)`, ensuring that every possible word ID corresponds to a valid row in the embedding matrix.

In [12]:
s2i = {'<UNK>': 0}
for i, w in enumerate(vocab):
  s2i[w] = i + 1
i2s = {i: w for w, i in s2i.items()}

In [13]:
vocab_size = len(s2i)
print("Vocab size:", vocab_size)

Vocab size: 6928


**5.** Define helper functions for converting between text and integer token IDs:

- `encode(text)`: converts a string into a list of integer token IDs using the `s2i` mapping. Tokens that are not in the vocabulary are mapped to `<UNK>` (ID `0`).
- `decode(ids)`: converts a list of token IDs back into a list of tokens using the `i2s` mapping.

These functions allow us to move between readable text and the integer IDs required for embedding learning and semantic analysis.

In [14]:
def encode(text):
  return [s2i.get(w, 0) for w in clean_text(text).split()]
def decode(ids):
  return [i2s[i] for i in ids]

In [15]:
sample = "the king and the queen are a man and a woman"
print(encode(sample))
print(decode(encode(sample)))
print(" ".join(decode(encode(sample))))

[1, 34, 2, 1, 89, 40, 8, 93, 2, 8, 454]
['the', 'king', 'and', 'the', 'queen', 'are', 'a', 'man', 'and', 'a', 'woman']
the king and the queen are a man and a woman


In [16]:
ids = encode(text)
print(ids[:100])

[88, 269, 139, 35, 968, 143, 668, 127, 15, 102, 33, 102, 102, 88, 269, 6, 40, 33, 1267, 350, 3, 199, 63, 3, 3331, 33, 1267, 1267, 88, 269, 88, 6, 91, 1140, 231, 11, 2273, 580, 3, 1, 305, 33, 35, 2554, 35, 2554, 88, 269, 70, 78, 502, 26, 2, 369, 23, 1339, 54, 39, 162, 2274, 642, 8, 3971, 33, 32, 52, 2874, 882, 70, 16, 17, 164, 148, 148, 166, 269, 72, 219, 46, 612, 88, 269, 35, 40, 5004, 173, 612, 1, 1894, 46, 28, 1268, 5005, 45, 56, 3972, 78, 38, 58, 56]


## Prepare the training dataset (Skip-gram with negative sampling)

In this step, we prepare the training data for the Skip-gram word embedding model. The Skip-gram objective is to predict surrounding context words given a center word. To do this, we construct:
- *Positive (center, context) word pairs* using a sliding window
- *Negative samples* for each positive pair to form the final training targets

**6.** Create Skip-gram (center, context) pairs

Given a sequence of word IDs, we generate training data by treating each word as a **center word** and pairing it with all nearby **context words** within a fixed window.

- The `window` size controls how many words to the left and right are considered context.
- For a single center word, **multiple** context words are generated.
- In the Skip-gram formulation, this one-to-many relationship is flattened into multiple (center, context) pairs.
- As a result, the same center word ID appears **repeatedly**, once for each of its context words.

The function returns two aligned arrays:
- `center_ids`: center word IDs (with repeated entries)
- `context_ids`: corresponding positive context word IDs

Each position `i` in `center_ids` and `context_ids` together forms **one training pair**.

For example, with `window = 2`:

Tokens: `[the, king, and, the, queen]`  
Center word: `king`  
Context words: `[the, and]`  

This produces the following training pairs:

- `(king, the)`
- `(king, and)`

In [17]:
def make_skipgram_pairs(ids, window=4):
    center_ids = []
    context_ids = []
    for i in range(len(ids)):
        for j in range(1, window + 1):
            if i - j >= 0:
                center_ids.append(ids[i])
                context_ids.append(ids[i - j])
            if i + j < len(ids):
                center_ids.append(ids[i])
                context_ids.append(ids[i + j])

    return np.array(center_ids), np.array(context_ids)

In [18]:
center_ids, context_ids = make_skipgram_pairs(ids, WINDOW)
print("Num positive pairs:", len(center_ids))
print("Example pairs (center, context):", [(decode([a])[0], decode([b])[0]) for a, b in zip(center_ids[:3], context_ids[:3])])

Num positive pairs: 1632476
Example pairs (center, context): [('first', 'citizen'), ('first', 'before'), ('first', 'we')]


**7.** Add negative samples

Computing a full softmax over the entire vocabulary is expensive. Instead, we use **negative sampling**, which turns the problem into a **binary** classification task.

For each positive (center, context) pair:
- We keep the true context word as a positive example
- We randomly sample `neg_k` negative words from the vocabulary
- Negative words are chosen so they are **not equal** to the center and true context word
- We also avoid sampling the `<UNK>` token as a negative example

This results in:
- One positive target
- Multiple negative targets

For each center word, we construct:
- `target_ids`: a list of `[positive_context, negative_1, ..., negative_k]`
- `labels`: corresponding labels `[1, 0, ..., 0]`

These will be used to train the Skip-gram model to:
- Assign **high similarity** to true context words
- Assign **low similarity** to unrelated (negative) words

In [19]:
def make_negative_pairs(center_ids, context_ids, vocab_size, neg_k=5):
    N = len(center_ids)
    pos = context_ids.reshape(N, 1)
    centers = center_ids.reshape(N, 1)

    # negatives: [N, neg_k]
    neg_ids = np.random.randint(1, vocab_size, size=(N, neg_k))  # avoid sampling <UNK> as negatives (id=0)
    mask = (neg_ids == pos) | (neg_ids == centers)
    while mask.any():
        neg_ids[mask] = np.random.randint(1, vocab_size, size=neg_ids[mask].shape)
        mask = (neg_ids == pos) | (neg_ids == centers)

    # targets: first column is positive, rest are negatives => [N, 1+neg_k]
    target_ids = np.concatenate([pos, neg_ids], axis=1)
    # labels: 1 for positive, 0 for negatives => [N, 1+neg_k]
    labels = np.zeros((N, 1 + neg_k))
    labels[:, 0] = 1
    return center_ids, target_ids, labels

In [20]:
# Demo: generate negative-sampling training data
train_center_ids, target_ids, labels = make_negative_pairs(center_ids, context_ids, vocab_size, neg_k=NEG_K)

# Sanity check: negatives should never equal the positive context (col 0)
assert not np.any(target_ids[:, 1:] == target_ids[:, [0]])
print("OK: negatives never equal the positive context.")

# (Optional) sanity: avoid sampling the center word as a negative
assert not np.any(target_ids[:, 1:] == train_center_ids.reshape(-1, 1))
print("OK: negatives never equal the center word.")

# Show a few examples
for i in range(3):
    center_word = decode([train_center_ids[i]])[0]
    target_words = decode(target_ids[i])   # list of words: [pos, neg1, neg2, ...]
    label_vals = labels[i]                 # [1, 0, 0, 0, ...]

    print(f"\nExample {i+1}:")
    print("  Center:", center_word)
    print("  Targets (pos | negatives):", target_words)
    print("  Labels:", label_vals)

OK: negatives never equal the positive context.
OK: negatives never equal the center word.

Example 1:
  Center: first
  Targets (pos | negatives): ['citizen', 'piteous', 'instinct', 'mean', 'sitting', 'practise']
  Labels: [1. 0. 0. 0. 0. 0.]

Example 2:
  Center: first
  Targets (pos | negatives): ['before', 'predecessors', 'knighthood', 'encounter', 'enjoying', 'creatures']
  Labels: [1. 0. 0. 0. 0. 0.]

Example 3:
  Center: first
  Targets (pos | negatives): ['we', 'fouler', 'bites', 'forty', 'hunt', 'differences']
  Labels: [1. 0. 0. 0. 0. 0.]


**Note:** The code below wraps the prepared arrays into a `tf.data.Dataset` for efficient training (shuffling, batching, and prefetching). It does not change the data or learning logic.

In [21]:
# center_ids: [N]
# target_ids: [N, K+1]
# labels:     [N, K+1]
# Use CPU to avoid GPU OOM when loading large arrays
with tf.device('/CPU:0'):
    ds = tf.data.Dataset.from_tensor_slices(((train_center_ids, target_ids), labels)) \
        .shuffle(200_000) \
        .batch(BATCH_SIZE) \
        .prefetch(tf.data.AUTOTUNE)

I0000 00:00:1772770779.332354 3190301 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 49180 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:01:00.0, compute capability: 8.0
I0000 00:00:1772770779.335136 3190301 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 21479 MB memory:  -> device: 1, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:25:00.0, compute capability: 8.0
I0000 00:00:1772770779.336620 3190301 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 79192 MB memory:  -> device: 2, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:c1:00.0, compute capability: 8.0
I0000 00:00:1772770779.338791 3190301 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:3 with 65367 MB memory:  -> device: 3, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:e1:00.0, compute capability: 8.0


## Define the Skip-gram model (Negative Sampling)

**7.** In this step, we define a Skip-gram model using a single **Embedding** layer. Conceptually, an embedding layer behaves like a linear layer applied to a one-hot input: it simply selects the corresponding row of an embedding matrix.

Given:
- `center_ids`: shape `[B]`
- `target_ids`: shape `[B, K+1]` (1 positive + K negatives)

The model looks up:
- the center embedding vector `v` with shape `[B, D]`
- the target embedding vectors `u` with shape `[B, K+1, D]`

It then computes a dot product between the center vector and each target vector to produce:
- `logits`: shape `[B, K+1]`

These logits will later be used with a binary classification loss so that the positive context receives a higher score than the negatives.

In [22]:
class SkipGramNS(keras.Model):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.embed = layers.Embedding(vocab_size, emb_dim)

    def call(self, inputs):
        center_ids, target_ids = inputs
        v = self.embed(center_ids)      # [B, D]
        u = self.embed(target_ids)      # [B, K+1, D]
        logits = tf.einsum("bd,bkd->bk", v, u)  # [B, K+1]
        return logits

In [23]:
model = SkipGramNS(vocab_size, EMB_DIM)

## Train the model (loss, compile, fit)

**8.** In this step, we train the Skip-gram model using the dataset prepared earlier. We use a binary classification loss (with `from_logits=True`) to train the embeddings so that:
- positive pairs get high dot-product scores
- negative pairs get low dot-product scores

We then `compile()` the model with an `Adam` optimizer and the loss, and call `fit()` to train.

In [24]:
loss = keras.losses.BinaryCrossentropy(from_logits=True)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss=loss, metrics=['accuracy'])

In [25]:
history = model.fit(ds, epochs=EPOCHS)

Epoch 1/20


2026-03-05 22:19:41.988325: I external/local_xla/xla/service/service.cc:163] XLA service 0x764e9c0026e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-05 22:19:41.988395: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA A100 80GB PCIe, Compute Capability 8.0
2026-03-05 22:19:41.988401: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (1): NVIDIA A100 80GB PCIe, Compute Capability 8.0
2026-03-05 22:19:41.988405: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (2): NVIDIA A100 80GB PCIe, Compute Capability 8.0
2026-03-05 22:19:41.988412: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (3): NVIDIA A100 80GB PCIe, Compute Capability 8.0
2026-03-05 22:19:42.012831: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-05 22:1

 20/798 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.2039 - loss: 0.6930

I0000 00:00:1772770782.634121 3191708 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


798/798 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.6337 - loss: 0.5147
Epoch 2/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.6416 - loss: 0.4940
Epoch 3/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.6437 - loss: 0.4912
Epoch 4/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.6478 - loss: 0.4881
Epoch 5/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.6525 - loss: 0.4849
Epoch 6/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.6581 - loss: 0.4819
Epoch 7/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.6628 - loss: 0.4786
Epoch 8/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.6669 - loss: 0.4754
Epoch 9/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - accuracy: 0.6703 - loss: 0.4723
Epoch 10/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.6734 - loss: 0.4690
Epoch 11/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.6760 - loss: 0.4659
Epoch 12/20
798/798 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accurac

## Cosine similarity and word embeddings

**9.** After training, the learned word embeddings are stored in the **embedding matrix** of the model. Each row corresponds to the vector representation of one word in the vocabulary.

We first extract this embedding matrix, then define helper functions to:
- `word2vec_map()`: map a word to its embedding vector, and
- `cosine_sim()`: measure semantic similarity between two words using **cosine similarity**.

Cosine similarity compares the *direction* of two vectors (rather than their magnitude), making it a standard choice for measuring similarity in embedding spaces.


In [26]:
emb_matrix = model.embed.get_weights()[0]  # shape: [vocab_size, EMB_DIM]
print("Embedding matrix shape:", emb_matrix.shape)

Embedding matrix shape: (6928, 128)


In [27]:
def word2vec_map(w): return emb_matrix[s2i.get(w, 0)]
def cosine_sim(a, b): return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

In [28]:
she = word2vec_map("she")
he = word2vec_map("he")
her = word2vec_map("her")

print("cosine_sim(she, he) =", cosine_sim(she, he))
print("cosine_sim(she, her) =", cosine_sim(she, her))

cosine_sim(she, he) = 0.3031402291901916
cosine_sim(she, her) = 0.4307222241433865


## Nearest neighbor search in embedding space

**10.** In this step, we explore the learned word embeddings by finding **nearest neighbors** in the embedding space. Given a query `word`, we retrieve the words whose embedding vectors are most similar to it, using cosine similarity.

The `nearest_neighbors` function takes the embedding matrix `emb_matrix` and vocabulary mappings (`s2i`, `i2s`) as input. This design allows the same function to be reused with **different embedding sources**, such as embeddings learned by our Skip-gram model or pre-trained embeddings.

For a given word, we:
- look up its embedding vector,
- compute cosine similarity with all other word vectors,
- sort by similarity score, and
- return the top-`k` most similar words (excluding the word itself).

In [29]:
def nearest_neighbors(word, emb_matrix, s2i, i2s, k=5):
    """Top-k nearest neighbors by cosine similarity (excluding the word itself)."""
    if word not in s2i:
        return []
    wid = s2i[word]
    v = emb_matrix[wid]

    norms = np.linalg.norm(emb_matrix, axis=1)
    sims = np.dot(emb_matrix, v) / (norms * np.linalg.norm(v) + 1e-8)

    best = np.argsort(-sims)[1:k+1]  # skip itself
    return [(i2s[i], float(sims[i])) for i in best]

In [30]:
# Nearest neighbors for a few example words
example_words = ["she", "he", "king", "queen", "doctor"]

for w in example_words:
    print(f"\nNearest neighbors for '{w}':")
    for nb, score in nearest_neighbors(w, emb_matrix, s2i, i2s, k=5):
        print(f"  {nb:>10s}  (cosine similarity = {score:.4f})")


Nearest neighbors for 'she':
         her  (cosine similarity = 0.4307)
        'tis  (cosine similarity = 0.4254)
    daughter  (cosine similarity = 0.4144)
        face  (cosine similarity = 0.4079)
   petruchio  (cosine similarity = 0.3955)

Nearest neighbors for 'he':
        that  (cosine similarity = 0.5515)
          is  (cosine similarity = 0.5181)
         not  (cosine similarity = 0.4851)
         for  (cosine similarity = 0.4666)
          as  (cosine similarity = 0.4568)

Nearest neighbors for 'king':
          if  (cosine similarity = 0.4736)
          iv  (cosine similarity = 0.4502)
          do  (cosine similarity = 0.4476)
     richard  (cosine similarity = 0.4416)
         not  (cosine similarity = 0.4275)

Nearest neighbors for 'queen':
    margaret  (cosine similarity = 0.5565)
   elizabeth  (cosine similarity = 0.5189)
     warwick  (cosine similarity = 0.4201)
       being  (cosine similarity = 0.3927)
      indeed  (cosine similarity = 0.3555)

Nearest neighbors

**Note:** The word `"doctor"` does not appear in the TinyShakespeare vocabulary, so it is treated as an out-of-vocabulary (OOV) word. As a result, nearest-neighbor search returns an empty list. This highlights an important limitation of embeddings trained on small or domain-specific datasets.

##  Word analogy analysis

**11.** In this step, we test whether the embedding space captures **semantic relationships** using *word analogies*. A classic example is:

> **king − man + woman ≈ queen**

The idea is that certain relationships (e.g., gender, tense, country–capital) can appear as approximately consistent **vector offsets** in a well-trained embedding space.

Given three words `(a, b, c)`, we form the **analogy query**:

$$
\text{query} = \mathbf{v}_b - \mathbf{v}_a + \mathbf{v}_c
$$

We then find the words whose embedding vectors are most similar to this query vector using cosine similarity.

**Important details:**
- If any of the input words are out-of-vocabulary (OOV), the function returns an empty list.
- We exclude the original words `(a, b, c)` from the results so the model does not trivially return them.
- As with nearest neighbors, we pass in `emb_matrix`, `s2i`, and `i2s` so the same function works for both our trained embeddings and pre-trained embeddings.

In [31]:
def analogy(a, b, c, emb_matrix, s2i, i2s, k=5):
    for w in (a, b, c):
        if w not in s2i:
            return []

    va = emb_matrix[s2i[a]]
    vb = emb_matrix[s2i[b]]
    vc = emb_matrix[s2i[c]]

    query = vb - va + vc

    norms = np.linalg.norm(emb_matrix, axis=1)
    sims = np.dot(emb_matrix, query) / (norms * np.linalg.norm(query) + 1e-8)

    for w in (a, b, c):
        sims[s2i[w]] = -np.inf

    best = np.argsort(-sims)[:k]
    return [(i2s[i], float(sims[i])) for i in best]

In [32]:
# Analogy examples: b - a + c ≈ ?
analogy_examples = [
    ("man", "king", "woman"),     # king - man + woman ≈ queen
    ("he", "king", "she"),
    ("man", "prince", "woman"),
    ("father", "king", "mother"),
]

for a, b, c in analogy_examples:
    print(f"\nAnalogy: {b} - {a} + {c} ≈ ?")
    results = analogy(a, b, c, emb_matrix, s2i, i2s, k=5)

    if not results:
        print("  (One or more words not in vocabulary)")
        continue

    for w, score in results:
        print(f"  {w:>10s}  (cosine similarity = {score:.4f})")


Analogy: king - man + woman ≈ ?
       lewis  (cosine similarity = 0.3211)
      heaven  (cosine similarity = 0.3195)
          vi  (cosine similarity = 0.3069)
          iv  (cosine similarity = 0.2871)
          do  (cosine similarity = 0.2795)

Analogy: king - he + she ≈ ?
          iv  (cosine similarity = 0.4400)
          vi  (cosine similarity = 0.3717)
      edward  (cosine similarity = 0.3456)
     richard  (cosine similarity = 0.3147)
     warwick  (cosine similarity = 0.3092)

Analogy: prince - man + woman ≈ ?
    florizel  (cosine similarity = 0.3359)
        post  (cosine similarity = 0.3211)
      heaven  (cosine similarity = 0.3152)
      london  (cosine similarity = 0.3103)
       title  (cosine similarity = 0.2997)

Analogy: king - father + mother ≈ ?
          iv  (cosine similarity = 0.4422)
          vi  (cosine similarity = 0.3407)
          if  (cosine similarity = 0.3088)
         iii  (cosine similarity = 0.3084)
         not  (cosine similarity = 0.3054)


## Load pre-trained GloVe embeddings and compare results

**12.** So far, we trained word embeddings on a small dataset (TinyShakespeare). Because the dataset is limited, many words may be missing from the vocabulary and analogy performance may be weak.

In this step, we load **pre-trained GloVe embeddings**, which were trained on a much larger corpus. We then repeat the same **nearest-neighbor** and **analogy** experiments using the pre-trained embedding matrix to observe improved semantic structure.

**What to do:**
1. Download and unzip the GloVe vectors.
2. Load the vectors into a dictionary: `word -> vector`.
3. Build the corresponding `s2i_pre`, `i2s_pre`, and `emb_pre`.
4. Run the same nearest-neighbor and analogy examples as before, but using the pre-trained embeddings.

**Note:** Pre-trained embeddings have their own vocabulary and mappings, so we construct separate variables:
- `s2i_pre`, `i2s_pre`, `emb_pre`

This is why our `nearest_neighbors(...)` and `analogy(...)` functions take `(emb_matrix, s2i, i2s)` as input.

In [33]:
!wget https://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2026-03-05 22:21:54--  https://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-03-05 22:21:54--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  4.92MB/s    in 2m 49s  

2026-03-05 22:24:43 (4.86 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflatin

In [34]:
def load_glove(path):
    word2vec = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split()
            word = parts[0]
            vec = np.array(parts[1:], dtype=np.float32)
            word2vec[word] = vec
    return word2vec

glove = load_glove("glove.6B.50d.txt")
print("Loaded words:", len(glove), "dim:", len(next(iter(glove.values()))))

Loaded words: 400000 dim: 50


In [35]:
words = list(glove.keys())
s2i_pre = {w: i for i, w in enumerate(words)}
i2s_pre = {i: w for w, i in s2i_pre.items()}
emb_pre = np.array([glove[w] for w in words], dtype=np.float32)

In [36]:
# Nearest neighbors using pretrained embeddings
for w in example_words:
    print(f"\n[GloVe] Nearest neighbors for '{w}':")
    results = nearest_neighbors(w, emb_pre, s2i_pre, i2s_pre, k=5)
    if not results:
        print("  (word not in GloVe vocabulary)")
        continue
    for nb, score in results:
        print(f"  {nb:>10s}  (cosine similarity = {score:.4f})")


[GloVe] Nearest neighbors for 'she':
         her  (cosine similarity = 0.9434)
     herself  (cosine similarity = 0.8923)
          he  (cosine similarity = 0.8852)
         him  (cosine similarity = 0.8597)
      mother  (cosine similarity = 0.8512)

[GloVe] Nearest neighbors for 'he':
         his  (cosine similarity = 0.9243)
        when  (cosine similarity = 0.9233)
         him  (cosine similarity = 0.9167)
      having  (cosine similarity = 0.9038)
        then  (cosine similarity = 0.9029)

[GloVe] Nearest neighbors for 'king':
      prince  (cosine similarity = 0.8236)
       queen  (cosine similarity = 0.7839)
          ii  (cosine similarity = 0.7746)
     emperor  (cosine similarity = 0.7736)
         son  (cosine similarity = 0.7667)

[GloVe] Nearest neighbors for 'queen':
    princess  (cosine similarity = 0.8515)
        lady  (cosine similarity = 0.8051)
   elizabeth  (cosine similarity = 0.7873)
        king  (cosine similarity = 0.7839)
      prince  (cosine similar

In [ ]:
# Analogy using pretrained embeddings
for a, b, c in analogy_examples:
    print(f"\n[GloVe] Analogy: {b} - {a} + {c} ≈ ?")
    results = analogy(a, b, c, emb_pre, s2i_pre, i2s_pre, k=5)

    if not results:
        print("  (One or more words not in GloVe vocabulary)")
        continue

    for w, score in results:
        print(f"  {w:>10s}  (cosine similarity = {score:.4f})")


[GloVe] Analogy: king - man + woman ≈ ?
       queen  (cosine similarity = 0.8610)
    daughter  (cosine similarity = 0.7685)
      prince  (cosine similarity = 0.7641)
      throne  (cosine similarity = 0.7635)
    princess  (cosine similarity = 0.7513)

[GloVe] Analogy: king - he + she ≈ ?
       queen  (cosine similarity = 0.8831)
    princess  (cosine similarity = 0.8046)
    daughter  (cosine similarity = 0.7792)
   elizabeth  (cosine similarity = 0.7728)
      prince  (cosine similarity = 0.7650)

[GloVe] Analogy: prince - man + woman ≈ ?
    princess  (cosine similarity = 0.8490)
       queen  (cosine similarity = 0.8173)
    daughter  (cosine similarity = 0.7722)
      eldest  (cosine similarity = 0.7636)
        wife  (cosine similarity = 0.7598)

[GloVe] Analogy: king - father + mother ≈ ?
       queen  (cosine similarity = 0.8776)
    princess  (cosine similarity = 0.7717)
      prince  (cosine similarity = 0.7535)
    daughter  (cosine similarity = 0.7417)
   elizabeth  (c

: 